# 03b — ProtoNet (Metric Learning)

**Paradigm:** Metric learning via prototypical networks (Snell et al., 2017)  
**Encoder:** CNN backbone loaded from `models/custom_cnn/best_model.pt` (frozen first, then fine-tuned)  
**Embedding:** 256-dim L2-normalised projection head  
**Episodes:** N=7-way, K=5-shot, Q=15 queries/class  
**Loss:** Prototypical cross-entropy + SupCon auxiliary (λ=0.1)  
**Curriculum:** Same three-phase SNR schedule as CNN (A→B→C)  
**Target:** macro F1 > 0.80, AUC-ROC > 0.85 per class

In [1]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import json, random, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler, ConcatDataset
from tqdm import tqdm
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    roc_auc_score, average_precision_score
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cuda


In [2]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
CFG = {
    # paths
    'data_root':       Path('/kaggle/input/datasets/orpheusmanga/alertreck-processed/processed/mel'),
    'cnn_checkpoint':  Path('/kaggle/input/datasets/orpheusmanga/alertreck-cnn-checkpoint/best_model.pt'),
    'output_dir':      Path('/kaggle/working/protonet'),
    # data
    'n_classes':       7,
    'n_mels':          128,
    'n_frames':        301,
    'label_names':     ['background_animals','background_wind_rain','threat_chainsaw',
                        'threat_dog','threat_gunshot','threat_human','threat_vehicle'],
    'seed':            42,
    'batch_size':      256,
    # episode design
    'n_way':           7,
    'k_shot':          5,
    'n_query':         15,
    'episodes_per_epoch': 200,
    # embedding
    'embed_dim':       256,
    # training
    'freeze_epochs':   5,
    'epochs':          50,
    'lr':              1e-4,
    'lr_head':         1e-3,
    'weight_decay':    1e-4,
    'patience':        12,
    'supcon_lambda':   0.1,
    'supcon_temp':     0.07,
    'phase_epochs':    {'A': 20, 'B': 15, 'C': 999},
}

CFG['output_dir'].mkdir(parents=True, exist_ok=True)

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
random.seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

print('CFG loaded.')
print(f'data_root : {CFG["data_root"]}')
print(f'checkpoint: {CFG["cnn_checkpoint"]}')


CFG loaded.
data_root : /kaggle/input/datasets/orpheusmanga/alertreck-processed/processed/mel
checkpoint: /kaggle/input/datasets/orpheusmanga/alertreck-cnn-checkpoint/best_model.pt


In [3]:
# ── 3. Dataset ────────────────────────────────────────────────────────────────
class ShardDataset(Dataset):
    """
    Loads all .npz shards from a split directory into memory.
    Each shard contains X (N, 128, n_frames) float32 and y (N,) int64.
    """
    def __init__(self, shard_dir: Path, augment: bool = False):
        shards = sorted(shard_dir.glob('*.npz'))
        assert shards, f'No shards found in {shard_dir}'

        xs, ys = [], []
        for s in tqdm(shards, desc=f'Loading {shard_dir.name}'):
            d = np.load(s)
            xs.append(d['X'])
            ys.append(d['y'].astype(np.int64))

        self.X      = np.concatenate(xs, axis=0)
        self.y      = np.concatenate(ys, axis=0)
        self.augment = augment
        print(f'  {shard_dir.name}: {len(self.X):,} samples')

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].copy()
        mean_val = x.mean()

        if self.augment:
            for _ in range(2):
                f_width = random.randint(0, 20)
                f0      = random.randint(0, x.shape[0] - f_width)
                x[f0:f0 + f_width, :] = mean_val
            for _ in range(2):
                t_width = random.randint(0, 40)
                t0      = random.randint(0, x.shape[1] - t_width)
                x[:, t0:t0 + t_width] = mean_val

        x = torch.from_numpy(x).unsqueeze(0)          # (1, 128, n_frames)
        y = torch.tensor(self.y[idx], dtype=torch.long)
        return x, y


data_root = CFG['data_root']

print('Loading val / test …')
val_ds  = ShardDataset(data_root / 'val',  augment=False)
test_ds = ShardDataset(data_root / 'test', augment=False)

print('\nLoading curriculum phases …')
train_clean_ds  = ShardDataset(data_root / 'train',       augment=True)
train_aug_A_ds  = ShardDataset(data_root / 'train_aug_A', augment=True)
train_aug_B_ds  = ShardDataset(data_root / 'train_aug_B', augment=True)
train_aug_C_ds  = ShardDataset(data_root / 'train_aug_C', augment=True)

# Phase datasets: clean + augmented (same as CNN notebook)
phase_datasets = {
    'A': ConcatDataset([train_clean_ds, train_aug_A_ds]),
    'B': ConcatDataset([train_clean_ds, train_aug_B_ds]),
    'C': ConcatDataset([train_clean_ds, train_aug_C_ds]),
}

val_loader  = DataLoader(val_ds,  batch_size=CFG['batch_size'], shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=4, pin_memory=True)

print(f'\nVal  : {len(val_ds):,} samples')
print(f'Test : {len(test_ds):,} samples')
print(f'Phase A train : {len(phase_datasets["A"]):,} samples')
print(f'Phase B train : {len(phase_datasets["B"]):,} samples')
print(f'Phase C train : {len(phase_datasets["C"]):,} samples')


Loading val / test …


Loading val: 100%|██████████| 6/6 [00:08<00:00,  1.37s/it]


  val: 5,844 samples


Loading test: 100%|██████████| 6/6 [00:07<00:00,  1.22s/it]


  test: 5,925 samples

Loading curriculum phases …


...

  train_aug_A: 14,854 samples


Loading train_aug_B: 100%|██████████| 30/30 [00:31<00:00,  1.04s/it]


  train_aug_B: 29,708 samples


Loading train_aug_C: 100%|██████████| 45/45 [00:38<00:00,  1.17it/s]


  train_aug_C: 44,562 samples

Val  : 5,844 samples
Test : 5,925 samples
Phase A train : 29,708 samples
Phase B train : 44,562 samples
Phase C train : 59,416 samples


In [4]:
# ── 4. Model: CNN Encoder + Projection Head ───────────────────────────────────

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(dropout),
        )
    def forward(self, x): return self.block(x)


class AudioCNN(nn.Module):
    """Exact replica of CNN notebook architecture — needed to load checkpoint."""
    def __init__(self, n_classes: int = 7):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(1,   32,  dropout=0.2),
            ConvBlock(32,  64,  dropout=0.2),
            ConvBlock(64,  128, dropout=0.2),
            ConvBlock(128, 256, dropout=0.2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.pool(self.encoder(x)).flatten(1))
    def encode(self, x):
        return self.pool(self.encoder(x)).flatten(1)


class ProtoNet(nn.Module):
    """
    CNN encoder (pretrained) + lightweight projection head.
    embed_dim: 256 → 256 with L2 normalisation.
    """
    def __init__(self, cnn_checkpoint: Path, embed_dim: int = 256, n_classes: int = 7):
        super().__init__()
        cnn = AudioCNN(n_classes=n_classes)
        ckpt = torch.load(cnn_checkpoint, map_location='cpu', weights_only=True)
        # checkpoint may be a full training snapshot or a raw state_dict
        state = ckpt.get('model_state', ckpt)
        cnn.load_state_dict(state)
        self.encoder = cnn.encoder
        self.pool    = cnn.pool

        self.proj = nn.Sequential(
            nn.Linear(256, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, embed_dim, bias=False),
        )

        self.register_buffer('prototypes', torch.zeros(n_classes, embed_dim))
        self.n_classes = n_classes
        self.embed_dim = embed_dim

    def embed(self, x):
        feat = self.pool(self.encoder(x)).flatten(1)
        return F.normalize(self.proj(feat), dim=1)

    def forward(self, x):
        return self.embed(x) @ self.prototypes.T

    def freeze_encoder(self):
        for p in self.encoder.parameters(): p.requires_grad_(False)
        for p in self.pool.parameters():    p.requires_grad_(False)

    def unfreeze_encoder(self):
        for p in self.encoder.parameters(): p.requires_grad_(True)
        for p in self.pool.parameters():    p.requires_grad_(True)


model = ProtoNet(
    cnn_checkpoint=CFG['cnn_checkpoint'],
    embed_dim=CFG['embed_dim'],
    n_classes=CFG['n_classes'],
).to(device)

model.freeze_encoder()

n_params    = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {n_params:,}')
print(f'Trainable params: {n_trainable:,}  (encoder frozen)')

dummy = torch.randn(4, 1, CFG['n_mels'], CFG['n_frames']).to(device)
emb   = model.embed(dummy)
print(f'Embed shape     : {tuple(emb.shape)}')
print(f'L2 norms        : {emb.norm(dim=1).tolist()}')


Total params    : 1,304,224
Trainable params: 131,584  (encoder frozen)
Embed shape     : (4, 256)
L2 norms        : [0.9999999403953552, 0.9999999403953552, 0.9999999403953552, 1.0]


In [5]:
# ── 5. Loss Functions ─────────────────────────────────────────────────────────

def prototypical_loss(support_emb, support_labels, query_emb, query_labels, n_way, k_shot):
    """
    Prototypical cross-entropy loss.
    support_emb : (n_way*k_shot, D) — L2-normalised
    query_emb   : (n_way*n_query, D)
    """
    protos = torch.stack([
        support_emb[support_labels == c].mean(0)
        for c in range(n_way)
    ])
    protos = F.normalize(protos, dim=1)

    logits = 10.0 * (query_emb @ protos.T)  # (n_query_total, n_way)

    unique  = torch.unique(support_labels, sorted=True)
    remap   = {c.item(): i for i, c in enumerate(unique)}
    local_q = torch.tensor([remap[l.item()] for l in query_labels],
                            dtype=torch.long, device=query_emb.device)

    loss = F.cross_entropy(logits, local_q)
    acc  = (logits.argmax(1) == local_q).float().mean().item()
    return loss, acc, logits, protos


def supcon_loss(embeddings, labels, temperature: float = 0.07):
    """
    Supervised contrastive loss (Khosla et al., 2020).
    embeddings : (N, D) — L2-normalised
    labels     : (N,)
    """
    N   = embeddings.size(0)
    sim = (embeddings @ embeddings.T) / temperature  # (N, N)

    labels   = labels.view(-1, 1)
    pos_mask = (labels == labels.T).float()
    pos_mask.fill_diagonal_(0)

    # numerically stable: subtract row max
    sim_max = sim.detach().max(dim=1, keepdim=True).values
    sim     = sim - sim_max

    exp_sim = torch.exp(sim)
    # zero out self-similarity WITHOUT in-place op (avoids autograd corruption)
    self_mask = ~torch.eye(N, dtype=torch.bool, device=embeddings.device)
    exp_sim   = exp_sim * self_mask.float()

    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)

    n_pos = pos_mask.sum(dim=1).clamp(min=1)
    loss  = -(pos_mask * log_prob).sum(dim=1) / n_pos
    return loss.mean()


print('Loss functions defined.')


Loss functions defined.


In [6]:
# ── 6. Episodic Sampler ───────────────────────────────────────────────────────

def build_class_index(dataset):
    """
    Build class → [indices] map.
    Works with ShardDataset (.y numpy array) and ConcatDataset of ShardDatasets.
    """
    idx_by_class = defaultdict(list)
    if hasattr(dataset, 'y'):
        # ShardDataset
        for i, label in enumerate(dataset.y):
            idx_by_class[int(label)].append(i)
    else:
        # ConcatDataset — walk each constituent dataset
        offset = 0
        for ds in dataset.datasets:
            for i, label in enumerate(ds.y):
                idx_by_class[int(label)].append(offset + i)
            offset += len(ds)
    return idx_by_class


class EpisodicBatchSampler(Sampler):
    """
    Yields one flat index list per episode:
    [class0_s0..class0_sK, class0_q0..class0_qQ, class1_s0..., ...]
    """
    def __init__(self, idx_by_class, n_way, k_shot, n_query, n_episodes):
        self.idx_by_class = idx_by_class
        self.n_way      = n_way
        self.k_shot     = k_shot
        self.n_query    = n_query
        self.n_episodes = n_episodes

    def __len__(self):
        return self.n_episodes

    def __iter__(self):
        for _ in range(self.n_episodes):
            classes = random.sample(list(self.idx_by_class.keys()), self.n_way)
            episode = []
            for c in classes:
                chosen = random.sample(self.idx_by_class[c], self.k_shot + self.n_query)
                episode.extend(chosen)
            yield episode


def make_episode_loader(dataset):
    idx = build_class_index(dataset)
    sampler = EpisodicBatchSampler(
        idx_by_class=idx,
        n_way=CFG['n_way'],
        k_shot=CFG['k_shot'],
        n_query=CFG['n_query'],
        n_episodes=CFG['episodes_per_epoch'],
    )
    return DataLoader(dataset, batch_sampler=sampler, num_workers=4, pin_memory=True)


def parse_episode(batch):
    """Split flat episodic batch into support / query tensors."""
    X, y   = batch
    N, K, Q = CFG['n_way'], CFG['k_shot'], CFG['n_query']
    X = X.view(N, K + Q, *X.shape[1:])
    y = y.view(N, K + Q)
    support_x = X[:, :K].reshape(N*K, *X.shape[2:]).to(device)
    query_x   = X[:, K:].reshape(N*Q, *X.shape[2:]).to(device)
    support_y = y[:, :K].reshape(N*K).to(device)
    query_y   = y[:, K:].reshape(N*Q).to(device)
    return support_x, support_y, query_x, query_y


print('Episodic sampler defined.')


Episodic sampler defined.


In [7]:
# ── 7. Optimiser & Scheduler ──────────────────────────────────────────────────

encoder_params    = list(model.encoder.parameters()) + list(model.pool.parameters())
projection_params = list(model.proj.parameters())

optimizer = optim.AdamW([
    {'params': projection_params, 'lr': CFG['lr_head']},
    {'params': encoder_params,    'lr': CFG['lr']},
], weight_decay=CFG['weight_decay'])

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=1e-6,
)

print('Optimiser: AdamW | head lr={:.0e} | encoder lr={:.0e}'.format(
      CFG['lr_head'], CFG['lr']))
print(f'Scheduler: CosineAnnealingLR T_max={CFG["epochs"]}')

Optimiser: AdamW | head lr=1e-03 | encoder lr=1e-04
Scheduler: CosineAnnealingLR T_max=50


## 7b — Three hyperparameter experiments (replaces Optuna)

We run **three explicit experiments**, each trained through the **full episodic curriculum** (encoder frozen for `freeze_epochs`, then fine-tuned), and keep the model with the highest **val macro-F1**:

| | encoder lr | head lr | weight_decay | embed_dim | supcon_λ | supcon_τ | k_shot |
|---|---|---|---|---|---|---|---|
| **A** baseline | 1e-4 | 1e-3 | 1e-4 | 256 | 0.10 | 0.07 | 5 |
| **B** more-SupCon | 1e-4 | 1e-3 | 1e-4 | 256 | 0.30 | 0.07 | 5 |
| **C** small-embed | 5e-5 | 1e-3 | 3e-4 | 128 | 0.10 | 0.10 | 10 |

The winner's checkpoint becomes `best_model.pt`, the comparison is saved to `experiments.json`, and the downstream eval/ONNX cells run on it unchanged.

> **Cost:** ProtoNet epochs are slow (~70 s frozen, ~155 s unfrozen), so three full curricula ≈ **4–5 h** on a T4. Drop an experiment or lower `CFG['epochs']` / `CFG['episodes_per_epoch']` if your session is tight.

In [8]:
# ── 7b. Three hyperparameter experiments ──────────────────────────────────────
# Each config is trained through the full episodic curriculum (next cell) with the
# fixed early-stop, and the best by VALIDATION macro-F1 is kept (experiments.json).
# 'lr' is the encoder fine-tuning LR, 'lr_head' the projection-head LR. Selection
# is on val only — the test set is untouched until the final evaluation.
EXPERIMENTS = [
    {'name': 'A_baseline',   'lr': 1e-4, 'lr_head': 1e-3, 'weight_decay': 1e-4,
     'embed_dim': 256, 'supcon_lambda': 0.10, 'supcon_temp': 0.07, 'k_shot': 5},
    {'name': 'B_moreSupCon', 'lr': 1e-4, 'lr_head': 1e-3, 'weight_decay': 1e-4,
     'embed_dim': 256, 'supcon_lambda': 0.30, 'supcon_temp': 0.07, 'k_shot': 5},
    {'name': 'C_smallEmbed', 'lr': 5e-5, 'lr_head': 1e-3, 'weight_decay': 3e-4,
     'embed_dim': 128, 'supcon_lambda': 0.10, 'supcon_temp': 0.10, 'k_shot': 10},
]

print(f'{len(EXPERIMENTS)} experiments — each runs the full episodic curriculum (~4–5 h total on T4):')
for e in EXPERIMENTS:
    print(f"  {e['name']:<14} " + "  ".join(f'{k}={v}' for k, v in e.items() if k != 'name'))

3 experiments — each runs the full episodic curriculum (~4–5 h total on T4):
  A_baseline     lr=0.0001  lr_head=0.001  weight_decay=0.0001  embed_dim=256  supcon_lambda=0.1  supcon_temp=0.07  k_shot=5
  B_moreSupCon   lr=0.0001  lr_head=0.001  weight_decay=0.0001  embed_dim=256  supcon_lambda=0.3  supcon_temp=0.07  k_shot=5
  C_smallEmbed   lr=5e-05  lr_head=0.001  weight_decay=0.0003  embed_dim=128  supcon_lambda=0.1  supcon_temp=0.1  k_shot=10


In [9]:
# ── 8. Train the 3 experiments (full curriculum) & keep the best ──────────────
import shutil, gc

best_model_path = CFG['output_dir'] / 'best_model.pt'


def get_phase(epoch):
    if epoch < CFG['phase_epochs']['A']:                             return 'A'
    if epoch < CFG['phase_epochs']['A'] + CFG['phase_epochs']['B']:  return 'B'
    return 'C'


def evaluate(loader):
    """Nearest-prototype classification metrics on `loader` (uses global model)."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits     = model(X)
            loss       = F.cross_entropy(logits, y)
            total_loss += loss.item() * len(y)
            n          += len(y)
            all_probs.append(F.softmax(logits, dim=1).cpu())
            all_preds.append(logits.argmax(1).cpu())
            all_labels.append(y.cpu())
    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    probs  = torch.cat(all_probs).numpy()
    f1     = f1_score(labels, preds, average='macro', zero_division=0)
    return total_loss / n, preds, labels, probs, f1


def update_prototypes(dataset):
    """Recompute class prototypes over a full dataset into the global model buffer."""
    model.eval()
    loader = DataLoader(dataset, batch_size=512, shuffle=False, num_workers=4, pin_memory=True)
    sums   = torch.zeros(CFG['n_classes'], CFG['embed_dim'], device=device)
    counts = torch.zeros(CFG['n_classes'], device=device)
    with torch.no_grad():
        for X, y in loader:
            z = model.embed(X.to(device))
            for c in range(CFG['n_classes']):
                mask = (y == c)
                if mask.any():
                    sums[c]   += z[mask].sum(0)
                    counts[c] += mask.sum()
    model.prototypes.copy_(F.normalize(sums / counts.unsqueeze(1).clamp(min=1), dim=1))


def run_experiment(exp):
    """Full episodic-curriculum training for one config. Rebuilds the global model
    + optimizer/scheduler so the helpers above operate on it. Saves a per-experiment
    state_dict and returns (best_val_f1, best_epoch, best_phase, history, ckpt_path)."""
    global model, optimizer, scheduler
    # episode/loss knobs read from CFG by make_episode_loader / parse_episode / update_prototypes
    CFG.update({'embed_dim': exp['embed_dim'], 'k_shot': exp['k_shot'],
                'supcon_lambda': exp['supcon_lambda'], 'supcon_temp': exp['supcon_temp']})

    model = ProtoNet(cnn_checkpoint=CFG['cnn_checkpoint'],
                     embed_dim=exp['embed_dim'], n_classes=CFG['n_classes']).to(device)
    model.freeze_encoder()
    enc = list(model.encoder.parameters()) + list(model.pool.parameters())
    optimizer = optim.AdamW([{'params': list(model.proj.parameters()), 'lr': exp['lr_head']},
                             {'params': enc, 'lr': exp['lr']}], weight_decay=exp['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)

    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_f1': [], 'lr': [], 'phase': []}
    best_f1, best_ep, best_ph, pat = 0.0, 0, 'A', 0
    ckpt_path = CFG['output_dir'] / f"model_{exp['name']}.pt"
    cur_phase = 'A'
    ep_loader = make_episode_loader(phase_datasets['A'])

    print(f"\n=== {exp['name']} === enc_lr={exp['lr']:.0e} head_lr={exp['lr_head']:.0e} "
          f"wd={exp['weight_decay']:.0e} embed={exp['embed_dim']} "
          f"sc_lam={exp['supcon_lambda']} sc_tmp={exp['supcon_temp']} k={exp['k_shot']}", flush=True)
    t0 = time.time()
    for epoch in range(CFG['epochs']):
        phase = get_phase(epoch)
        if phase != cur_phase:
            cur_phase = phase
            ep_loader = make_episode_loader(phase_datasets[phase])
            pat = 0          # give each curriculum phase its own early-stop budget
        if epoch == CFG['freeze_epochs']:
            model.unfreeze_encoder()

        model.train()
        ep_losses, ep_accs = [], []
        for batch in ep_loader:
            sup_x, sup_y, qry_x, qry_y = parse_episode(batch)
            sup_z, qry_z = model.embed(sup_x), model.embed(qry_x)
            proto_loss, ep_acc, _, _ = prototypical_loss(
                sup_z, sup_y, qry_z, qry_y, CFG['n_way'], CFG['k_shot'])
            sc_loss = supcon_loss(torch.cat([sup_z, qry_z]),
                                  torch.cat([sup_y, qry_y]), CFG['supcon_temp'])
            loss = proto_loss + CFG['supcon_lambda'] * sc_loss
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            ep_losses.append(loss.item()); ep_accs.append(ep_acc)
        scheduler.step()

        update_prototypes(phase_datasets[phase])
        va_loss, _, _, _, va_f1 = evaluate(val_loader)

        for k, v in [('train_loss', float(np.mean(ep_losses))), ('train_acc', float(np.mean(ep_accs))),
                     ('val_loss', va_loss), ('val_f1', va_f1),
                     ('lr', optimizer.param_groups[0]['lr']), ('phase', phase)]:
            hist[k].append(v)

        flag = ''
        if va_f1 > best_f1:
            best_f1, best_ep, best_ph, pat = va_f1, epoch + 1, phase, 0
            torch.save(model.state_dict(), ckpt_path)
            flag = ' ✓'
        else:
            pat += 1
        print(f"  [{phase}] ep {epoch+1:02d}  va_f1={va_f1:.4f} (best {best_f1:.4f})"
              f"  ({time.time()-t0:.0f}s){flag}", flush=True)
        # only early-stop in the final phase so the full A->B->C curriculum always runs
        if pat >= CFG['patience'] and phase == 'C':
            print(f"  early stop at epoch {epoch+1}", flush=True)
            break

    print(f"  → {exp['name']} best val macro-F1 = {best_f1:.4f} "
          f"(epoch {best_ep}, phase {best_ph}, {(time.time()-t0)/60:.1f} min)", flush=True)
    return best_f1, best_ep, best_ph, hist, ckpt_path


exp_results = []
for exp in EXPERIMENTS:
    bf1, bep, bph, hist, ckpt = run_experiment(exp)
    exp_results.append({'name': exp['name'], 'val_f1': bf1, 'epoch': bep, 'phase': bph,
                        'ckpt': str(ckpt), 'history': hist, 'exp': exp})
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

# ── pick the winner and adopt it ──────────────────────────────────────────────
best = max(exp_results, key=lambda r: r['val_f1'])
shutil.copy(best['ckpt'], best_model_path)

print("\n" + "=" * 56)
print("Experiment comparison (val macro-F1):")
for r in sorted(exp_results, key=lambda r: -r['val_f1']):
    mark = '   <-- BEST' if r['name'] == best['name'] else ''
    print(f"  {r['name']:<14} {r['val_f1']:.4f}  (epoch {r['epoch']}, {r['phase']}){mark}")
print(f"\nBest model → {best_model_path}  ({best['name']}, val_F1={best['val_f1']:.4f})")
print("=" * 56)

# adopt the winning config + set the globals the downstream cells expect
CFG.update(best['exp'])
history     = best['history']
best_val_f1 = best['val_f1']
best_epoch  = best['epoch']
best_phase  = best['phase']
model = ProtoNet(cnn_checkpoint=CFG['cnn_checkpoint'],
                 embed_dim=CFG['embed_dim'], n_classes=CFG['n_classes']).to(device)
model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))

(CFG['output_dir'] / 'experiments.json').write_text(json.dumps(
    [{'name': r['name'], 'val_f1': r['val_f1'], 'epoch': r['epoch'], 'phase': r['phase'],
      'config': r['exp']} for r in exp_results], indent=2))
print(f"Comparison saved → {CFG['output_dir'] / 'experiments.json'}")


=== A_baseline === enc_lr=1e-04 head_lr=1e-03 wd=1e-04 embed=256 sc_lam=0.1 sc_tmp=0.07 k=5
  [A] ep 01  va_f1=0.8037 (best 0.8037)  (83s) ✓
  [A] ep 02  va_f1=0.7886 (best 0.8037)  (168s)
  [A] ep 03  va_f1=0.8030 (best 0.8037)  (252s)
  [A] ep 04  va_f1=0.8013 (best 0.8037)  (337s)
  [A] ep 05  va_f1=0.8021 (best 0.8037)  (421s)
  [A] ep 06  va_f1=0.8073 (best 0.8073)  (587s) ✓
  [A] ep 07  va_f1=0.7996 (best 0.8073)  (753s)
  [A] ep 08  va_f1=0.8125 (best 0.8125)  (919s) ✓
  [A] ep 09  va_f1=0.8120 (best 0.8125)  (1085s)
  [A] ep 10  va_f1=0.8032 (best 0.8125)  (1251s)
  [A] ep 11  va_f1=0.8039 (best 0.8125)  (1417s)
  [A] ep 12  va_f1=0.8122 (best 0.8125)  (1582s)
  [A] ep 13  va_f1=0.8162 (best 0.8162)  (1748s) ✓
  [A] ep 14  va_f1=0.8101 (best 0.8162)  (1914s)
  [A] ep 15  va_f1=0.8027 (best 0.8162)  (2080s)
  [A] ep 16  va_f1=0.8272 (best 0.8272)  (2246s) ✓
  [A] ep 17  va_f1=0.8101 (best 0.8272)  (2412s)
  [A] ep 18  va_f1=0.8166 (best 0.8272)  (2578s)
  [A] ep 19  va_f1=0.830

In [10]:
# ── 9. Training Curves ────────────────────────────────────────────────────────
epochs_ran = len(history['train_loss'])
xs = range(1, epochs_ran + 1)

# Phase boundary shading
phase_colors = {'A': '#e8f5e9', 'B': '#fff9c4', 'C': '#fce4ec'}
phase_names  = history['phase']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

def shade_phases(ax):
    prev_p, prev_e = phase_names[0], 1
    for i, p in enumerate(phase_names):
        if p != prev_p or i == epochs_ran - 1:
            end_e = i + 1 if p != prev_p else i + 2
            ax.axvspan(prev_e - 0.5, end_e - 0.5, alpha=0.35,
                       color=phase_colors[prev_p], label=f'Phase {prev_p}')
            prev_p, prev_e = p, i + 1

# Loss
shade_phases(axes[0])
axes[0].plot(xs, history['train_loss'], label='Train loss')
axes[0].plot(xs, history['val_loss'],   label='Val loss')
axes[0].set(title='Episode Loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend()

# Episode accuracy (train)
shade_phases(axes[1])
axes[1].plot(xs, history['train_acc'], label='Episode acc (train)')
axes[1].set(title='Episode Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[1].axhline(0.80, ls='--', color='grey', alpha=0.5, label='0.80 target')
axes[1].legend()

# Val macro F1
shade_phases(axes[2])
axes[2].plot(xs, history['val_f1'], color='purple', label='Val macro F1')
axes[2].axhline(best_val_f1, ls='--', color='green', alpha=0.6,
                label=f'Best {best_val_f1:.4f}')
axes[2].axvline(best_epoch, ls=':', color='green', alpha=0.6)
axes[2].set(title='Val Macro F1', xlabel='Epoch', ylabel='F1')
axes[2].legend()

plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'training_curves.png', dpi=150)
print('Saved training_curves.png')

Saved training_curves.png


In [11]:
# ── 10. Test Evaluation ───────────────────────────────────────────────────────
state = torch.load(CFG['output_dir'] / 'best_model.pt', map_location=device, weights_only=True)
model.load_state_dict(state)

# Recompute prototypes on full Phase-C training set
update_prototypes(phase_datasets['C'])
print('Prototypes recomputed on Phase-C training set.')

te_loss, te_preds, te_labels, te_probs, te_f1 = evaluate(test_loader)
te_acc = (te_preds == te_labels).mean()

print(f'\nTest accuracy   : {te_acc:.4f}')
print(f'Test macro F1   : {te_f1:.4f}')
print(f'\n{classification_report(te_labels, te_preds, target_names=CFG["label_names"], digits=4)}')


Prototypes recomputed on Phase-C training set.

Test accuracy   : 0.8241
Test macro F1   : 0.8036

                      precision    recall  f1-score   support

  background_animals     0.8714    0.7749    0.8204      1373
background_wind_rain     0.8996    0.7580    0.8227      1855
     threat_chainsaw     0.7516    0.8848    0.8128       547
          threat_dog     0.5760    0.9191    0.7082       235
      threat_gunshot     0.8282    0.8580    0.8428      1000
        threat_human     0.8221    0.9828    0.8953       696
      threat_vehicle     0.6732    0.7808    0.7230       219

            accuracy                         0.8241      5925
           macro avg     0.7746    0.8512    0.8036      5925
        weighted avg     0.8370    0.8241    0.8249      5925



In [12]:
# ── 11. Confusion Matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(te_labels, te_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(CFG['n_classes'])); ax.set_yticks(range(CFG['n_classes']))
ax.set_xticklabels(CFG['label_names'], rotation=45, ha='right')
ax.set_yticklabels(CFG['label_names'])
ax.set(xlabel='Predicted', ylabel='True', title='Confusion Matrix (row-normalised) — ProtoNet')
for i in range(CFG['n_classes']):
    for j in range(CFG['n_classes']):
        ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                color='white' if cm_norm[i,j] > 0.6 else 'black', fontsize=8)
plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'confusion_matrix.png', dpi=150)
print('Saved confusion_matrix.png')

Saved confusion_matrix.png


In [13]:
# ── 12. AUC-ROC + Precision/Recall Evaluation ─────────────────────────────────
from sklearn.metrics import roc_curve, precision_recall_curve

classes = list(range(CFG['n_classes']))
y_bin   = label_binarize(te_labels, classes=classes)  # (N, 7)

per_class_auc = {}
per_class_ap  = {}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# ROC curves
for c, name in enumerate(CFG['label_names']):
    fpr, tpr, _ = roc_curve(y_bin[:, c], te_probs[:, c])
    auc = roc_auc_score(y_bin[:, c], te_probs[:, c])
    per_class_auc[name] = auc
    axes[0].plot(fpr, tpr, label=f'{name} ({auc:.3f})')
axes[0].axvline(0.20, ls='--', color='grey', alpha=0.5, label='FPR=20% limit')
axes[0].plot([0,1],[0,1], 'k--', alpha=0.3)
axes[0].set(xlabel='FPR', ylabel='TPR', title='ROC Curves (one-vs-rest) — ProtoNet')
axes[0].legend(fontsize=8)

# PR curves
for c, name in enumerate(CFG['label_names']):
    prec, rec, _ = precision_recall_curve(y_bin[:, c], te_probs[:, c])
    ap = average_precision_score(y_bin[:, c], te_probs[:, c])
    per_class_ap[name] = ap
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.3f})')
axes[1].set(xlabel='Recall', ylabel='Precision', title='PR Curves — ProtoNet')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(CFG['output_dir'] / 'roc_pr_curves.png', dpi=150)
print('Saved roc_pr_curves.png')

macro_auc = np.mean(list(per_class_auc.values()))
macro_ap  = np.mean(list(per_class_ap.values()))
print(f'\nMacro AUC-ROC : {macro_auc:.4f}  (target >0.85)')
print(f'Macro AP      : {macro_ap:.4f}')
print('\nPer-class AUC-ROC:')
for name, auc in sorted(per_class_auc.items(), key=lambda x: x[1]):
    flag = '  OK' if auc >= 0.85 else '  BELOW TARGET'
    print(f'  {name:<30} {auc:.4f}{flag}')

Saved roc_pr_curves.png

Macro AUC-ROC : 0.9748  (target >0.85)
Macro AP      : 0.9072

Per-class AUC-ROC:
  background_wind_rain           0.9518  OK
  threat_gunshot                 0.9670  OK
  background_animals             0.9725  OK
  threat_vehicle                 0.9762  OK
  threat_chainsaw                0.9779  OK
  threat_dog                     0.9832  OK
  threat_human                   0.9951  OK


In [14]:
# ── 13. ONNX Export ───────────────────────────────────────────────────────────
onnx_path = CFG['output_dir'] / 'protonet.onnx'

model.eval()
dummy_input = torch.randn(1, 1, CFG['n_mels'], CFG['n_frames']).to(device)

torch.onnx.export(
    model, dummy_input, onnx_path,
    dynamo=False,         # legacy TorchScript exporter — no onnxscript dep
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['mel_spectrogram'],
    output_names=['class_logits'],
    dynamic_axes={
        'mel_spectrogram': {0: 'batch_size'},
        'class_logits':    {0: 'batch_size'},
    },
)
print(f'ONNX model exported → {onnx_path}')
print(f'File size: {onnx_path.stat().st_size / 1e6:.2f} MB')

/tmp/ipykernel_58/1674059861.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX model exported → /kaggle/working/protonet/protonet.onnx
File size: 5.23 MB


In [15]:
# ── 14. Save Results ──────────────────────────────────────────────────────────
from sklearn.metrics import precision_score, recall_score

per_class_f1   = f1_score(te_labels, te_preds, average=None, zero_division=0)
per_class_prec = precision_score(te_labels, te_preds, average=None, zero_division=0)
per_class_rec  = recall_score(te_labels, te_preds, average=None, zero_division=0)

results = {
    # metadata
    'model':         'ProtoNet',
    'best_epoch':    best_epoch,
    'best_phase':    best_phase,
    'best_val_f1':   best_val_f1,
    'n_params':      sum(p.numel() for p in model.parameters()),
    # CFG snapshot
    'n_way':         CFG['n_way'],
    'k_shot':        CFG['k_shot'],
    'n_query':       CFG['n_query'],
    'embed_dim':     CFG['embed_dim'],
    'supcon_lambda': CFG['supcon_lambda'],
    # experiment selection
    'best_config':   best['exp'],
    'experiments':   json.loads((CFG['output_dir'] / 'experiments.json').read_text())
                     if (CFG['output_dir'] / 'experiments.json').exists() else None,
    # test metrics
    'test_acc':      float(te_acc),
    'test_macro_f1': float(te_f1),
    'macro_auc':     float(macro_auc),
    'macro_ap':      float(macro_ap),
    # per-class
    'per_class_f1':  {n: float(v) for n, v in zip(CFG['label_names'], per_class_f1)},
    'per_class_prec':{n: float(v) for n, v in zip(CFG['label_names'], per_class_prec)},
    'per_class_rec': {n: float(v) for n, v in zip(CFG['label_names'], per_class_rec)},
    'per_class_auc': per_class_auc,
    'per_class_ap':  per_class_ap,
    # training history
    'history':       history,
}

with open(CFG['output_dir'] / 'results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Also save model config for cross-notebook reference
cfg_save = {
    'n_classes': CFG['n_classes'],
    'n_mels':    CFG['n_mels'],
    'n_frames':  CFG['n_frames'],
    'embed_dim': CFG['embed_dim'],
    'label_names': CFG['label_names'],
    'test_acc':   float(te_acc),
    'test_macro_f1': float(te_f1),
    'macro_auc':  float(macro_auc),
    'best_epoch': best_epoch,
    'best_val_f1': best_val_f1,
    'best_phase': best_phase,
}
with open(CFG['output_dir'] / 'model_config_protonet.json', 'w') as f:
    json.dump(cfg_save, f, indent=2)

print('Results saved to models/protonet/')
print(f"  best config   : {best['exp']['name']}")
print(f"  test_acc      : {te_acc:.4f}")
print(f"  test_macro_f1 : {te_f1:.4f}")
print(f"  macro_auc     : {macro_auc:.4f}")
print(f"  best_val_f1   : {best_val_f1:.4f}  (epoch {best_epoch}, Phase {best_phase})")

Results saved to models/protonet/
  best config   : A_baseline
  test_acc      : 0.8241
  test_macro_f1 : 0.8036
  macro_auc     : 0.9748
  best_val_f1   : 0.8303  (epoch 19, Phase A)
